In [52]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str


In [53]:
from dotenv import load_dotenv
import os
load_dotenv()
mlflow_tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
mlflow_tracking_uri

'https://dagshub.com/irinyenikanibukun/Full-Stack-ML-Project.mlflow'

In [58]:
from dsProject.constants import *
from dsProject.utils.common import read_yaml, create_directories
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifact_root])
        
    def get_model_evaluation_config(self)-> ModelEvaluationConfig:
        config = self.config.model_evaluation
        schema = self.schema
        params = self.params
        create_directories([config.root_dir])
        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            metric_file_name=config.metric_file_name,
            target_column=schema.TARGET_COLUMN.name,
            mlflow_uri=mlflow_tracking_uri,
            all_params=params.ElasticNet
        )
        return model_evaluation_config
        
        

In [59]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
from dsProject.utils.common import *
from urllib.parse import urlparse
import mlflow

class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
        
    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = load_bin(Path(self.config.model_path))
        
        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            predicted_qualites = model.predict(test_x)
            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualites)
            
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(Path(self.config.metric_file_name), scores)
            mlflow.log_params(self.config.all_params)
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)
            
            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            else:
                mlflow.sklearn.log_model(model, "model")
            
        

In [ ]:
# import dagshub
# dagshub.init(repo_owner='irinyenikanibukun', repo_name='Full-Stack-ML-Project', mlflow=True)

# import mlflow
# with mlflow.start_run():
#   mlflow.log_param('parameter name', 'value')
#   mlflow.log_metric('metric name', 1)
#   0121f27b2d7fac2655317961e16ace609b43dc9e

In [60]:
config  = ConfigurationManager()
model_evaluation_config = config.get_model_evaluation_config()
model_evaluation = ModelEvaluation(model_evaluation_config)
model_evaluation.log_into_mlflow()

2026-07-29 17:46:47,803 | INFO | common| YAML file: config\config.yaml loaded successfully.
2026-07-29 17:46:47,803 | INFO | common| YAML file: params.yaml loaded successfully.
2026-07-29 17:46:47,812 | INFO | common| YAML file: schema.yaml loaded successfully.
2026-07-29 17:46:47,813 | INFO | common| Directory created at: artifacts
2026-07-29 17:46:47,815 | INFO | common| Directory created at: artifacts/model_evaluation
2026-07-29 17:46:47,820 | INFO | common| Binary file loaded from: artifacts\model_trainer\model.joblib
2026-07-29 17:46:50,123 | INFO | common| JSON file saved at: artifacts\model_evaluation\metrics.json


2026/07/29 17:46:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/29 17:47:03 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Ibk\AppData\Local\Temp\tmpkrz4rpuz\model\model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.9.0', 'skops==0.14.0']. Set logging level to DEBUG to see the full traceback. 
2026/07/29 17:47:05 WARNING mlflow.utils.requirements_utils: Encountered an unexpected error (KeyError('Name')) while detecting model dependency mismatches. Set logging level to DEBUG to see the full traceback.
Successfully registered model 'ElasticnetModel'.
2026/07/29 17:47:11 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModel, version 1
Created version '1' of model 'ElasticnetModel'.


🏃 View run casual-vole-759 at: https://dagshub.com/irinyenikanibukun/Full-Stack-ML-Project.mlflow/#/experiments/0/runs/8322bae87a60498682eaf805c8b06281
🧪 View experiment at: https://dagshub.com/irinyenikanibukun/Full-Stack-ML-Project.mlflow/#/experiments/0


In [31]:
save_json?

Signature:       save_json(path: pathlib.Path, data: dict)
Call signature:  save_json(*args, **kwargs)
Type:            WrappedFunction
String form:     <function save_json at 0x0000029FBAEAC4A0>
File:            c:\users\ibk\desktop\data project\full-stack-ml-project\venv\lib\site-packages\ensure\main.py
Docstring:      
Saves a dictionary as a JSON file.
Args:
    path (Path): Path to save the JSON file.
    data (dict): Dictionary to save as JSON.
Class docstring: Wrapper for functions to check argument annotations

'https://dagshub.com/irinyenikanibukun/Full-Stack-ML-Project.mlflow'

In [12]:
# os.chdir("../")
!pwd

/c/Users/Ibk/Desktop/data project/Full-Stack-ML-Project
